# get historical daily urls 

In [0]:
import requests
import re

base_url = "https://www.nemweb.com.au/REPORTS/ARCHIVE/DispatchIS_Reports/"

html = requests.get(base_url).text

files = re.findall(#get list from the html
    r"PUBLIC_DISPATCHIS_\d{8}\.zip",#finds all text with this pattern (0-9, 8 times)
    html
)

urls = [
    base_url + file
    for file in sorted(set(files))
]

In [0]:
urls

# download files from url

In [0]:
import os
import time
import requests

def download_if_not_exists(url, bronze_folder):
    filename = url.split("/")[-1]
    path = f"{bronze_folder}/{filename}"

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")

    start = time.time()

    data = requests.get(url).content

    with open(path, "wb") as file:
        file.write(data)

    print(f"Finished: {filename} - {time.time() - start:.1f} seconds")

    return path

In [0]:
bronze_folder = "/Volumes/workspace/default/aemo_mlops_volume/bronze/daily"

os.makedirs(bronze_folder, exist_ok=True)

for url in urls:
    download_if_not_exists(url, bronze_folder)

# uncompressed csv files

In [0]:
import os
import zipfile
import time

csv_folder = bronze_folder + "_uncompressed"

os.makedirs(csv_folder, exist_ok=True)

zip_files = [
    file for file in os.listdir(bronze_folder)
    if file.endswith(".zip")
]

start = time.time()

print(f"Found {len(zip_files)} ZIP files\n")

for i, filename in enumerate(sorted(zip_files), 1):

    path = os.path.join(bronze_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:

        files = zip_file.namelist()

        already_extracted = all(
            os.path.exists(os.path.join(csv_folder, file))
            for file in files
        )

        if already_extracted:
            print(f"[{i}/{len(zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(zip_files)}] Extracting: {filename}")

        zip_file.extractall(csv_folder)

print()
print(f"Finished in {time.time() - start:.1f} seconds")